# Creación de vistas para Dashboard

Este notebook crea la capa semántica utilizada por el dashboard del proyecto Smart Data.

Las vistas consumen principalmente las tablas Gold y exponen métricas y dimensiones
preparadas para visualización.

Este notebook se ejecuta como parte del setup de la solución y no forma parte
del Job principal de procesamiento.

In [0]:
dbutils.widgets.text(
    "catalog_name",
    "smart_data_dev",
    "Catalog"
)

dbutils.widgets.text(
    "schema_name",
    "storage",
    "Schema"
)

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

print(f"Catalog: {catalog_name}")
print(f"Schema:  {schema_name}")

In [0]:
tabla_maestro_visitante = f"{catalog_name}.{schema_name}.maestro_visitante"
tabla_visitas_mensuales = f"{catalog_name}.{schema_name}.visitas_mensuales"
tabla_visitante_mensual = f"{catalog_name}.{schema_name}.visitante_mensual"
tabla_errores = f"{catalog_name}.{schema_name}.errores"
tabla_control_archivos = f"{catalog_name}.{schema_name}.control_archivos"

In [0]:
tablas = [
    tabla_maestro_visitante,
    tabla_visitas_mensuales,
    tabla_visitante_mensual,
    tabla_errores,
    tabla_control_archivos
]

for tabla in tablas:
    if not spark.catalog.tableExists(tabla):
        raise Exception(f"No existe la tabla requerida: {tabla}")

print("Todas las tablas requeridas existen.")

In [0]:
vista_resumen_mensual = f"{catalog_name}.{schema_name}.vw_dashboard_resumen_mensual"

spark.sql(f"""
CREATE OR REPLACE VIEW {vista_resumen_mensual} AS

SELECT
    periodo,
    TO_DATE(CONCAT(periodo, '-01')) AS periodo_fecha,

    COUNT(DISTINCT email) AS total_visitantes,

    COUNT(
        DISTINCT CASE
            WHEN periodo_baja IS NULL
            THEN email
        END
    ) AS visitantes_activos,

    COUNT(
        DISTINCT CASE
            WHEN periodo_baja IS NOT NULL
            THEN email
        END
    ) AS visitantes_baja,

    COUNT(
        DISTINCT CASE
            WHEN DATE_FORMAT(fecha_primera_visita, 'yyyy-MM') = periodo
            THEN email
        END
    ) AS visitantes_nuevos,

    COUNT(
        DISTINCT CASE
            WHEN visitas_mes_actual > 0
            THEN email
        END
    ) AS visitantes_con_actividad,

    COALESCE(SUM(visitas_mes_actual), 0) AS visitas_mes,

    COALESCE(SUM(visitas_anio_actual), 0) AS visitas_anio_acumuladas,

    COALESCE(SUM(visitas_totales), 0) AS visitas_totales_acumuladas,

    MAX(fecha_ejecucion) AS fecha_actualizacion

FROM {tabla_visitante_mensual}

GROUP BY
    periodo
""")

print(f"Vista creada: {vista_resumen_mensual}")

In [0]:
vista_actividad_mensual = f"{catalog_name}.{schema_name}.vw_dashboard_actividad_mensual"

spark.sql(f"""
CREATE OR REPLACE VIEW {vista_actividad_mensual} AS

SELECT
    periodo,
    TO_DATE(CONCAT(periodo, '-01')) AS periodo_fecha,

    COUNT(DISTINCT email) AS visitantes_con_actividad,

    COALESCE(SUM(visitas_mes), 0) AS total_visitas,

    ROUND(
        AVG(visitas_mes),
        2
    ) AS promedio_visitas_por_visitante,

    MIN(fecha_primera_visita_mes) AS primera_visita_mes,

    MAX(fecha_ultima_visita_mes) AS ultima_visita_mes,

    MAX(fecha_ejecucion) AS fecha_actualizacion

FROM {tabla_visitas_mensuales}

GROUP BY
    periodo
""")

print(f"Vista creada: {vista_actividad_mensual}")

In [0]:
vista_visitantes = f"{catalog_name}.{schema_name}.vw_dashboard_visitantes"

spark.sql(f"""
CREATE OR REPLACE VIEW {vista_visitantes} AS

SELECT
    periodo,
    TO_DATE(CONCAT(periodo, '-01')) AS periodo_fecha,

    email,
    fecha_primera_visita,
    fecha_ultima_visita,

    visitas_totales,
    visitas_anio_actual,
    visitas_mes_actual,

    periodo_baja,

    CASE
        WHEN periodo_baja IS NULL THEN 'Activo'
        ELSE 'Baja'
    END AS estado_visitante,

    fecha_ejecucion AS fecha_actualizacion

FROM {tabla_visitante_mensual}
""")

print(f"Vista creada: {vista_visitantes}")

In [0]:
vista_procesamiento = f"{catalog_name}.{schema_name}.vw_dashboard_procesamiento"

spark.sql(f"""
CREATE OR REPLACE VIEW {vista_procesamiento} AS

SELECT
    id_proceso,

    MAX(fecha_ejecucion) AS fecha_ejecucion,

    CAST(MAX(fecha_ejecucion) AS DATE) AS fecha_ejecucion_dia,

    COUNT(DISTINCT nombre_archivo) AS archivos_procesados,

    COALESCE(SUM(registros_bronze), 0) AS registros_bronze,

    COALESCE(SUM(registros_invalidos_bronze), 0) AS registros_invalidos_bronze,

    COALESCE(SUM(registros_silver), 0) AS registros_silver,

    COALESCE(SUM(registros_error), 0) AS registros_error

FROM {tabla_control_archivos}

GROUP BY
    id_proceso
""")

print(f"Vista creada: {vista_procesamiento}")

In [0]:
vista_errores = f"{catalog_name}.{schema_name}.vw_dashboard_errores"

spark.sql(f"""
CREATE OR REPLACE VIEW {vista_errores} AS

SELECT
    id_proceso,
    fecha_ejecucion,

    CAST(fecha_ejecucion AS DATE) AS fecha_ejecucion_dia,

    nombre_archivo,
    razon_error,

    COUNT(*) AS total_errores

FROM {tabla_errores}

GROUP BY
    id_proceso,
    fecha_ejecucion,
    nombre_archivo,
    razon_error
""")

print(f"Vista creada: {vista_errores}")

In [0]:
vistas = [
    vista_resumen_mensual,
    vista_actividad_mensual,
    vista_visitantes,
    vista_procesamiento,
    vista_errores
]

for vista in vistas:
    if not spark.catalog.tableExists(vista):
        raise Exception(f"No se creó correctamente la vista: {vista}")

print("Todas las vistas del dashboard fueron creadas correctamente.")